# Today it was sunny

In [18]:
import pandas as pd
import pandapower as pp
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import copy
import copy
import pulp
import time
import pulp
import csv
import random

#MY CODE BASE
import GridFlowSmooth as flow 

datelabel = str(datetime.now())[2:10]

# PandaPower Solver as Reference
"joint development of the research group of the Department for Sustainable Electrical Energy Systems (e2n), University of Kassel and the Department for Distribution System Operation at the Fraunhofer Institute for Energy Economics and Energy System Technology (IEE), Kassel." 
https://pandapower.readthedocs.io/en/latest/


In [32]:
def panda_DCOPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 dc_lines = [], generation_cost = {1: 120}, printout = True):
    net = pp.create_empty_network()
    ## # BUSES # ##
    pd_buses = {}             
    for i in buses:
        pd_buses[i] = pp.create_bus(net, vn_kv=1, name=f"Bus {i}")
    ## # LINES # ##
    for fb, tb in ac_lines:
        pp.create_line_from_parameters(net, from_bus=pd_buses[fb], to_bus=pd_buses[tb], length_km=1,
                                       r_ohm_per_km=0, x_ohm_per_km=1/susceptances[(fb,tb)], c_nf_per_km=0,
                                       max_i_ka=1000, name=f"Line {fb}-{tb}")
    ## # LOADS # ##
    for i in demands:
        pp.create_load(net, bus=pd_buses[i], p_mw=demands[i], q_mvar=0, controllable=False, name=f"Load Bus {i}")
    pd_gens = {}
    ## # GENERATORS # ##
    for i in gen_capacity:
        if gen_capacity[i]>0:
            pd_gens[i] = pp.create_gen(net, bus=pd_buses[i], p_mw=0, vm_pu=1.02, min_p_mw=0, max_p_mw=gen_capacity[i],
                         controllable=True, name=f"Gen Bus {i}", slack = len(pd_gens)==0 )
    for i in pd_gens:
        pp.create_poly_cost(net, element=pd_gens[i], et="gen", cp1_eur_per_mw=generation_cost[i])
    # Run DC OPF
    out =pp.runopp(net, delta=1e-10, calculate_voltage_angles=True, trafo_model="pi")
    if printout:
        print("\n--- Generator Results ---")
        print(net.res_gen)
        
        print("\n--- Line Flow ---")
        print(net.res_line[["p_from_mw"]])
        
        print("\n--- Bus Voltage Angles (rad) ---")
        print(net.res_bus["va_degree"]*np.pi/180)
    return net

In [34]:
prob = flow.ACDC_TEP_OPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 poss_ac_lines = [(1, 3), (1,4)], dc_lines = [], poss_dc_lines = [(3,1)], 
                 generation_cost = {1: 120}, printout = True,
                 max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)

Status: Optimal
Objective Value: 10800.0

Generation at Each Bus:
Bus 1: 90.0 MW

AC Line Flows:
Line 1-2: 60.0 MW
Line 2-3: 40.0 MW
Line 3-4: 20.0 MW
Line 1-3: 0.0 MW
Line 1-4: 0.0 MW

New AC Line Decisions:
Line 1-3: Not Added
Line 1-4: Not Added

New DC Line Decisions:
Line 3-1: Not Added

Voltage phases:
Bus 1: 0.0 rads
Bus 2: -0.04 rads
Bus 3: -0.066666667 rads
Bus 4: -0.08 rads


In [36]:
net = panda_DCOPF()


--- Generator Results ---
   p_mw    q_mvar  va_degree     vm_pu
0  90.0  3.737019        0.0  1.001559

--- Line Flow ---
   p_from_mw
0       60.0
1       40.0
2       20.0

--- Bus Voltage Angles (rad) ---
0    0.000000
1   -0.039954
2   -0.066645
3   -0.079997
Name: va_degree, dtype: float64


# Now let's run these a few times

In [ ]:
gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
 poss_ac_lines = [(1, 3), (1,4)], dc_lines = [], poss_dc_lines = [(3,1)], 
 generation_cost = {1: 120}, printout = True,
 max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False


prob = flow.ACDC_TEP_OPF(gen_capacity = {1:150,2:0,3:0,4:0}, line_capacity = 100, 
                 susceptances = {(1,2):1500, (2,3):1500, (3,4):1500, (1,3):1500, (1,4):1500}, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4], obj = 'cost', cut = False,
                 demands = {1: 30, 2: 20, 3: 20, 4: 20}, ac_lines = [(1, 2), (2,3), (3,4)], 
                 poss_ac_lines = [(1, 3), (1,4)], dc_lines = [], poss_dc_lines = [(3,1)], 
                 generation_cost = {1: 120}, printout = True,
                 max_new_ac = 0, max_new_dc = 0, label=datelabel, save = False)